# 第十二章

  
## 编译器和解释器
命令式编程很方便，但可能效率不高。  
符号式编程，即代码通常只在完全定义了过程之后才执行计算。  一般包括以下步骤：
1. 定义计算流程；
2. 将流程编译成可执行的程序；
3. 给定输入，调用编译好的程序执行。  

命令式编程更容易使用。符号式编程运行效率更高，更易于移植。 

## 异步计算
深度学习框架可以将Python前端的控制与后端的执行解耦，使得命令可以快速地异步插入后端、并行执行。
异步产生了一个相当灵活的前端，但过度填充任务队列可能会导致内存消耗过多。建议对每个小批量进行同步，以保持前端和后端大致同步。

  
## 自动并行
数据并行 vs 模型并行
数据并行:将小批量分成n块，每个GPU拿到完整参数计算一块数据的梯度
* 通常性能更好
模型并行:将模型分成n块，每个GPU拿到一块模型计算它的前向和方向结果
* 通常用于模型大到单GPU放不下
当一个模型能用单卡计算时，通常使用数据并行拓展到多卡上
模型并行则用在超大模型上 

现代系统还拥有各种通信资源，如PCI Express、存储（通常是固态硬盘或网络存储）和网络带宽，为了达到最高效率可以并行使用它们。

后端可以通过自动化地并行计算和通信来提高性能。

  
## 硬件
CPU:可以处理通用计算。性能优化考虑数据读写效率和多线程  
GPU:使用更多的小核和更好的内存带宽，适合能大规模并行的计算任务  
随机访问存储的一些关键特性是 带宽（bandwidth）和 延迟（latency）。  
硬盘的主要优点之一是相对便宜，而它们的缺点是典型的灾难性故障模式和相对较高的读取延迟。
固态驱动器（solid state drives，SSD）使用闪存持久地存储信息。这允许更快地访问存储的记录。固态驱动器以块的方式（256KB或更大）存储信息。块只能作为一个整体来写入，因此需要耗费大量的时间，导致固态驱动器在按位随机写入时性能非常差。固态驱动器中的存储单元磨损得比较快（通常在几千次写入之后就已经老化了）。  
寄存器是CPU可以以时钟速度访问而没有延迟的存储位置。一级缓存很小（常见的大小可能是32-64KB），内容通常分为数据和指令。二级缓存比一级缓存大（通常每个核心256-512KB），而速度也更慢。三级缓存在多个核之间共享，并且可以非常大。  
PCIe，一种专用总线，用于每个通道点到点连接的高带宽需求（在16通道插槽中的PCIe4.0上高达32GB/s），延迟时间为个位数的微秒（5μs）。  
将算法与硬件相匹配（例如，内存占用和带宽）。将命中参数装入缓存后，可以实现很大数量级的加速比。 

## 多 GPU 训练
数据并行通过增加有效的小批量数据量的大小提高了训练效率。
在数据并行中，数据需要跨多个GPU拆分，其中每个GPU执行自己的前向传播和反向传播，随后所有的梯度被聚合为一，之后聚合结果向所有的GPU广播。
小批量数据量更大时，学习率也需要稍微提高一些。  
每台设备上的网络需要先初始化，然后再尝试访问该设备上的参数。  
优化算法在多个GPU上自动聚合。
  
## 参数服务器（Parameter Server）
同步需要高度适应特定的网络基础设施和服务器内的连接，这种适应会严重影响同步所需的时间。
环同步对于p3和DGX-2服务器是最佳的，而对于其他服务器则未必。
当添加多个参数服务器以增加带宽时，分层同步策略可以工作的很好。  